In [7]:
import warnings
import os
import pandas as pd

# Suppressing pandas warnings
warnings.filterwarnings('ignore')

# Data source: RAND HRS Longitudinal File 2022
# Download link: https://hrsdata.isr.umich.edu/data-products/rand

def pull_target_columns(file_path):
# Reads the RAND dataset and extracts only the demographic and financial columns 
# Read just the first row to grab the column names (prevents memory crashes)
    sample = next(pd.read_stata(file_path, chunksize=1))
    available_cols = set(sample.columns)

    # Base demographics plus wave-level variables (waves 1-16)
    target_vars = ['hhidpn', 'ragender', 'rabyear', 'raedyrs', 'raeduc', 'raracem']
    for w in range(1, 17):
        target_vars += [f'r{w}work', f'r{w}shlt', f'h{w}atotb', f'r{w}agey_e', f'r{w}isret']

    # Only load columns that actually exist in the dataset to avoid KeyErrors
    cols_to_load = [c for c in target_vars if c in available_cols]
    print(f"Loading {len(cols_to_load)} of {len(target_vars)} target columns "
          f"(some variables are not collected in every wave).")

    return pd.read_stata(file_path, columns=cols_to_load)

# 1. Resolve the data path
if os.path.exists('data/randhrs1992_2022v1.dta'):
    data_path = 'data/randhrs1992_2022v1.dta'
elif os.path.exists('../data/randhrs1992_2022v1.dta'):
    data_path = '../data/randhrs1992_2022v1.dta'
else:
    raise FileNotFoundError("Could not find the RAND .dta file. Check the data/ folder.")

# 2. Execute the pull
df = pull_target_columns(data_path)

# 3. Print diagnostics and save the data for Notebook 01
print("Raw data shape:", df.shape)
print(df.head(3))

# Ensure output directory exists before saving
os.makedirs('../output', exist_ok=True)
df.to_csv('../output/00_pulled_data.csv', index=False)
print("Saved to output/00_pulled_data.csv")

Loading 86 of 86 target columns (some variables are not collected in every wave).
Raw data shape: (45234, 86)
   hhidpn  ragender  rabyear raedyrs                  raeduc  \
0    1010    1.male   1938.0    16.0     5.college and above   
1    2010  2.female   1934.0     8.0        1.lt high-school   
2    3010    1.male   1936.0    12.0  3.high-school graduate   

             raracem                 r1work  r1shlt   h1atotb  r1agey_e  ...  \
0  1.white/caucasian      1.working for pay  4.fair    6000.0      54.0  ...   
1  1.white/caucasian  0.not working for pay  3.good       0.0      57.0  ...   
2  1.white/caucasian      1.working for pay  4.fair  155000.0      56.0  ...   

   r15work r15shlt h15atotb  r15agey_e  r15isret  r16work r16shlt h16atotb  \
0      NaN     NaN      NaN        NaN       NaN      NaN     NaN      NaN   
1      NaN     NaN      NaN        NaN       NaN      NaN     NaN      NaN   
2      NaN     NaN      NaN        NaN       NaN      NaN     NaN      NaN   
